# Qwen2.5-Coder — NL → Java → C# Inference Notebook

**Model:** `shibsankardhara2/Qwen2.5-Coder-1.5B-Java-CSharp_V2`

This notebook loads the fine-tuned model directly from Hugging Face and runs inference only — **no training code**.

**Pipeline:**
```
Natural Language  ──(Stage 1)──▶  Java  ──(Stage 2)──▶  C#
```


In [1]:
!pip install -q \
    transformers>=4.40.0 \
    accelerate>=0.29.0 \
    bitsandbytes>=0.43.0 \
    peft>=0.10.0 \
    huggingface_hub>=0.22.0

print("All packages installed successfully.")

All packages installed successfully.


In [2]:
import gc
import json
import os
import time
from typing import Dict, Optional


import torch


from transformers import AutoModelForCausalLM, AutoTokenizer


from peft import PeftConfig, PeftModel


from IPython.display import Markdown, display

print(f"torch {torch.__version__} | CUDA available: {torch.cuda.is_available()}")

torch 2.10.0+cu128 | CUDA available: True


In [4]:
# ── Device detection ──────────────────────────────────────────────────────────
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Active device : {DEVICE}")
print(f"PyTorch version: {torch.__version__}")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    total_vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    cuda_ver = torch.version.cuda
    print(f"GPU            : {gpu_name}")
    print(f"Total VRAM     : {total_vram:.2f} GB")
    print(f"CUDA version   : {cuda_ver}")

    # Choose the best floating-point dtype for this GPU.
    # bfloat16 is preferred on Ampere+ (A100, RTX 3090+); fp16 on T4/V100.
    USE_BF16 = torch.cuda.is_bf16_supported()
    TORCH_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16
    print(f"Using dtype    : {TORCH_DTYPE}")
else:
    TORCH_DTYPE = torch.float32
    print("Running on CPU — inference will be slow.")


def log_gpu_memory(tag: str = "") -> None:
    """Print current GPU allocated/reserved memory (no-op on CPU)."""
    if not torch.cuda.is_available():
        return
    alloc = torch.cuda.memory_allocated() / 1e9
    reserved = torch.cuda.memory_reserved() / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    label = f"[{tag}] " if tag else ""
    print(f"GPU memory {label}| allocated: {alloc:.2f} GB  "
          f"| reserved: {reserved:.2f} GB  | total: {total:.2f} GB")


log_gpu_memory("startup")

Active device : cuda
PyTorch version: 2.10.0+cu128
GPU            : Tesla T4
Total VRAM     : 15.64 GB
CUDA version   : 12.8
Using dtype    : torch.bfloat16
GPU memory [startup] | allocated: 0.00 GB  | reserved: 0.00 GB  | total: 15.64 GB


In [5]:
# ── Model identifier ──────────────────────────────────────────────────────────
MODEL_ID = "shibsankardhara2/Qwen2.5-Coder-1.5B-Java-CSharp_V2"


def _is_lora_repo(repo_id: str) -> bool:
    """Return True if the Hub repo contains an adapter_config.json (LoRA repo)."""
    try:
        PeftConfig.from_pretrained(repo_id)
        return True
    except Exception:
        return False


def load_tokenizer_and_model(model_id: str):
    """
    Load the tokenizer and model from Hub.

    Auto-detects LoRA vs merged model:
      - LoRA  → load base model from adapter_config, then merge_and_unload().
      - Merged → load directly via AutoModelForCausalLM.

    Returns: (tokenizer, model)
    """
    print(f"Loading tokenizer from '{model_id}' ...")
    try:
        tok = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    except Exception as exc:
        raise RuntimeError(f"Tokenizer load failed: {exc}") from exc

    # Ensure pad token is defined (required for batched generation).
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token

    # ── Auto-detect adapter vs merged ────────────────────────────────────────
    if _is_lora_repo(model_id):
        # The repo contains LoRA adapter weights; retrieve the base model name
        # from adapter_config.json and merge the adapter before inference.
        print(f"'{model_id}' is a LoRA adapter repo.")
        peft_cfg = PeftConfig.from_pretrained(model_id)
        base_id = peft_cfg.base_model_name_or_path
        print(f"Loading base model '{base_id}' ...")
        try:
            base_model = AutoModelForCausalLM.from_pretrained(
                base_id,
                torch_dtype=TORCH_DTYPE,
                device_map="auto",
                low_cpu_mem_usage=True,
                trust_remote_code=True,
            )
        except Exception as exc:
            raise RuntimeError(f"Base model load failed: {exc}") from exc

        print("Merging LoRA adapter into base model weights ...")
        mdl = PeftModel.from_pretrained(base_model, model_id)
        mdl = mdl.merge_and_unload()   # fuse adapter; returns a plain nn.Module
        print("LoRA adapter merged and unloaded.")
    else:
        # The repo is already a merged / full-weight model.
        print(f"'{model_id}' is a merged model. Loading directly ...")
        try:
            mdl = AutoModelForCausalLM.from_pretrained(
                model_id,
                torch_dtype=TORCH_DTYPE,
                device_map="auto",
                low_cpu_mem_usage=True,
                trust_remote_code=True,
            )
        except Exception as exc:
            raise RuntimeError(f"Model load failed: {exc}") from exc

    mdl.eval()   # disable dropout / batch-norm training mode
    print("Model loaded and set to eval mode.")
    log_gpu_memory("after model load")
    return tok, mdl


# ── Load ──────────────────────────────────────────────────────────────────────
tokenizer, model = load_tokenizer_and_model(MODEL_ID)

Loading tokenizer from 'shibsankardhara2/Qwen2.5-Coder-1.5B-Java-CSharp_V2' ...


`torch_dtype` is deprecated! Use `dtype` instead!


'shibsankardhara2/Qwen2.5-Coder-1.5B-Java-CSharp_V2' is a merged model. Loading directly ...


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

Model loaded and set to eval mode.
GPU memory [after model load] | allocated: 1.50 GB  | reserved: 1.52 GB  | total: 15.64 GB


## 5. Print Model Information

Display key facts about the loaded model: parameter counts, dtype, device placement, and
current GPU memory usage.

In [6]:
# ── Parameter counts ─────────────────────────────────────────────────────────
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel()
                       for p in model.parameters() if p.requires_grad)

# ── Device placement (device_map="auto" may split across CPU/GPU) ─────────────
try:
    model_device = next(model.parameters()).device
except StopIteration:
    model_device = "unknown"

# ── Dtype ─────────────────────────────────────────────────────────────────────
try:
    model_dtype = next(model.parameters()).dtype
except StopIteration:
    model_dtype = "unknown"

print("=" * 55)
print("  MODEL INFORMATION")
print("=" * 55)
print(f"  Model ID         : {MODEL_ID}")
print(f"  Total parameters : {total_params:,}  ({total_params / 1e9:.3f} B)")
print(f"  Trainable params : {trainable_params:,}")
print(f"  Dtype            : {model_dtype}")
print(f"  Device           : {model_device}")
print("=" * 55)
log_gpu_memory("model info")

  MODEL INFORMATION
  Model ID         : shibsankardhara2/Qwen2.5-Coder-1.5B-Java-CSharp_V2
  Total parameters : 1,543,714,304  (1.544 B)
  Trainable params : 1,543,714,304
  Dtype            : torch.bfloat16
  Device           : cuda:0
GPU memory [model info] | allocated: 1.50 GB  | reserved: 1.52 GB  | total: 15.64 GB


## 6. Prompt Template Builders

The model was fine-tuned with specific prompt formats. These builders **must** match the
formats used during training exactly — any deviation degrades output quality.

- **Stage 1** (NL → Java): `### Instruction: … ### Response:\n\n`
- **Stage 2** (Java → C#): `### Instruction\n … ### Java\n … ### Response\n`

In [7]:
# ── Response marker constants (used to strip the prompt echo from output) ─────
# These MUST match the markers used during fine-tuning exactly.
STAGE1_RESPONSE_MARKER = "### Response:\n\n"
STAGE2_RESPONSE_MARKER = "### Response\n"

# Language hint appended to NL instructions so the model stays anchored to Java.
_JAVA_HINT = " Write the solution in Java."
# Language hint for C# stage.
_CSHARP_HINT = " Write the solution in C#."


def build_stage1_prompt(nl_prompt: str) -> str:
    """
    Build the Stage 1 (NL → Java) inference prompt.

    Format (must match training):
        ### Instruction:\n\n{nl_prompt} Write the solution in Java.\n\n### Response:\n\n
    """
    instruction = nl_prompt.strip()
    # Append the Java hint unless the prompt already mentions Java.
    if "java" not in instruction.lower():
        instruction = instruction + _JAVA_HINT
    return f"### Instruction:\n\n{instruction}\n\n{STAGE1_RESPONSE_MARKER}"


def build_stage2_prompt(java_code: str) -> str:
    """
    Build the Stage 2 (Java → C#) inference prompt.

    Format (must match training):
        ### Instruction\n{instruction}\n\n### Java\n{java_code}\n\n### Response\n
    """
    instruction = "Translate the following Java code into equivalent C#." + _CSHARP_HINT
    return (
        f"### Instruction\n{instruction}\n\n"
        f"### Java\n{java_code.strip()}\n\n"
        f"{STAGE2_RESPONSE_MARKER}"
    )


# ── Quick sanity check ────────────────────────────────────────────────────────
print("Stage 1 prompt sample:")
print(build_stage1_prompt("write a function to reverse a string"))
print("-" * 60)
print("Stage 2 prompt sample:")
print(build_stage2_prompt('public int add(int a, int b) { return a + b; }'))

Stage 1 prompt sample:
### Instruction:

write a function to reverse a string Write the solution in Java.

### Response:


------------------------------------------------------------
Stage 2 prompt sample:
### Instruction
Translate the following Java code into equivalent C#. Write the solution in C#.

### Java
public int add(int a, int b) { return a + b; }

### Response



In [19]:
import re

_STOP_SEQUENCES = (
    "### Instruction",
    "### Response",
    "### Java",
    "### C#",
    "### Input",
    "### Output",
    "<|endoftext|>",
    "<|im_end|>",
)


def _truncate_at_stop(text: str) -> str:
    """Cut ``text`` at the first stop marker so trailing rambling is removed."""
    cut = len(text)
    for marker in _STOP_SEQUENCES:
        pos = text.find(marker)
        if pos != -1:
            cut = min(cut, pos)
    return text[:cut].strip()


def _strip_code_fences(text: str) -> str:
    """Remove a surrounding ```lang ... ``` markdown fence if the model added one."""
    stripped = text.strip()
    if stripped.startswith("```"):
        # Drop the opening fence line (e.g. ```java) ...
        stripped = stripped.split("\n", 1)[-1] if "\n" in stripped else ""
        # ... and the closing fence, if present.
        if "```" in stripped:
            stripped = stripped[: stripped.rfind("```")]
    return stripped.strip()


def _clean_csharp_output(code: str) -> str:
    """
    Remove spurious `virtual` / `override` modifiers the model adds to bare
    (class-less) method snippets.

    The model was trained on C# methods that usually live inside a class, where
    `public virtual ...` is common. When it translates a *standalone* Java method
    it carries the `virtual` keyword over — but `virtual`/`override` are only
    valid on members of a class, so on a bare snippet they are invalid C#.
    We therefore strip them ONLY when the snippet has no enclosing type
    declaration (class / struct / interface / record / enum).
    """
    if re.search(r"\b(class|struct|interface|record|enum)\b", code):
        return code  # real type present — leave modifiers intact
    # Drop 'virtual'/'override' after an access modifier: 'public virtual int' -> 'public int'.
    code = re.sub(r"\b(public|private|protected|internal)\s+(?:virtual|override)\s+",
                  r"\1 ", code)
    # Drop a leading 'virtual'/'override' with no access modifier.
    code = re.sub(r"(^|\n)(\s*)(?:virtual|override)\s+", r"\1\2", code)
    return code


@torch.inference_mode()
def _generate(
    prompt_text: str,
    response_marker: str,
    max_new_tokens: int = 512,
) -> str:
    """
    Core greedy-decoding generation primitive.

    Args:
        prompt_text      : Full formatted prompt string.
        response_marker  : Marker used to strip the prompt echo from decoded text.
        max_new_tokens   : Maximum tokens to generate (default 512).

    Returns:
        Only the model's answer — prompt echo removed and any trailing extra
        section (repeated instructions / examples) truncated.
    """
    # Tokenize — keep tensors on the same device as the model.
    inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)
    input_len = inputs["input_ids"].shape[-1]

    try:
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            # ── Deterministic decoding ─────────────────────────────────────
            do_sample=False,      # greedy decoding
            temperature=1.0,      # no effect when do_sample=False; kept for clarity
            top_p=1.0,            # no nucleus sampling
            repetition_penalty=1.1,  # mild penalty to avoid repetitive output
            # ── Token IDs ──────────────────────────────────────────────────
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
        )
    except Exception as exc:
        raise RuntimeError(f"model.generate() failed: {exc}") from exc

    # Decode ONLY the newly generated tokens (everything after the prompt). This
    # avoids re-parsing the prompt echo and is robust even if the response marker
    # happens to appear inside the prompt text itself.
    generated_ids = output_ids[0][input_len:]
    answer = tokenizer.decode(generated_ids, skip_special_tokens=True)

    # Defensive: if the marker still leaked into the generated text, keep the part
    # after the LAST marker (the actual answer).
    if response_marker in answer:
        answer = answer.split(response_marker)[-1]

    # Truncate at the first new section marker, then strip any markdown fences.
    answer = _truncate_at_stop(answer)
    answer = _strip_code_fences(answer)
    return answer.strip()


# ── Public API ────────────────────────────────────────────────────────────────

def generate_java(nl_prompt: str, max_new_tokens: int = 512) -> str:
    """
    Stage 1: Natural Language → Java.

    Args:
        nl_prompt      : Plain-English description of the desired function.
        max_new_tokens : Maximum tokens to generate.

    Returns:
        Generated Java source code as a string.
    """
    if not isinstance(nl_prompt, str) or not nl_prompt.strip():
        raise ValueError("nl_prompt must be a non-empty string.")
    prompt = build_stage1_prompt(nl_prompt)
    try:
        return _generate(prompt, STAGE1_RESPONSE_MARKER, max_new_tokens)
    except Exception as exc:
        raise RuntimeError(f"generate_java failed: {exc}") from exc


def generate_csharp(java_code: str, max_new_tokens: int = 512) -> str:
    """
    Stage 2: Java → C#.

    Args:
        java_code      : Java source code to translate.
        max_new_tokens : Maximum tokens to generate.

    Returns:
        Generated C# source code as a string.
    """
    if not isinstance(java_code, str) or not java_code.strip():
        raise ValueError("java_code must be a non-empty string.")
    prompt = build_stage2_prompt(java_code)
    try:
        csharp = _generate(prompt, STAGE2_RESPONSE_MARKER, max_new_tokens)
        # Strip spurious 'virtual'/'override' modifiers from bare method snippets.
        return _clean_csharp_output(csharp)
    except Exception as exc:
        raise RuntimeError(f"generate_csharp failed: {exc}") from exc


def generate_end_to_end(
    nl_prompt: str,
    max_new_tokens: int = 512,
) -> Dict[str, object]:
    """
    Full pipeline: Natural Language → Java → C#.

    Args:
        nl_prompt      : Plain-English description of the desired function.
        max_new_tokens : Maximum tokens per stage.

    Returns:
        dict with keys:
            natural_language  – original prompt
            java              – generated Java code
            csharp            – generated C# code
            java_time_sec     – Stage 1 wall-clock time
            csharp_time_sec   – Stage 2 wall-clock time
            total_time_sec    – combined time
    """
    # Stage 1: NL → Java
    t0 = time.perf_counter()
    java_code = generate_java(nl_prompt, max_new_tokens=max_new_tokens)
    java_time = time.perf_counter() - t0

    # Stage 2: Java → C#
    t1 = time.perf_counter()
    csharp_code = generate_csharp(java_code, max_new_tokens=max_new_tokens)
    csharp_time = time.perf_counter() - t1

    return {
        "natural_language": nl_prompt,
        "java":             java_code,
        "csharp":           csharp_code,
        "java_time_sec":    round(java_time,  2),
        "csharp_time_sec":  round(csharp_time, 2),
        "total_time_sec":   round(java_time + csharp_time, 2),
    }


print("Inference helpers defined: generate_java(), generate_csharp(), generate_end_to_end()")

Inference helpers defined: generate_java(), generate_csharp(), generate_end_to_end()


In [ ]:
print("Interactive NL → Java → C# generator.")
print("Type a prompt and press Enter. Empty line / 'quit' / 'exit' to stop.\n")

while True:
    try:
        user_prompt = input("Your prompt > ").strip()
    except (EOFError, KeyboardInterrupt):
        print("\nInput stream closed. Exiting interactive loop.")
        break

    if not user_prompt or user_prompt.lower() in {"quit", "exit"}:
        print("Exiting interactive loop.")
        break

    try:
        result = generate_end_to_end(user_prompt)
    except Exception as exc:
        print(f"  ⚠  ERROR: {exc}\n")
        continue

    md = f"""
### Your Prompt: {result['natural_language']}

**Generated Java**
```java
{result['java']}
```

**Generated C#**
```csharp
{result['csharp']}
```

> ⏱ Java: **{result['java_time_sec']}s** | C#: **{result['csharp_time_sec']}s** | Total: **{result['total_time_sec']}s**

---
"""
    display(Markdown(md))

Interactive NL → Java → C# generator.
Type a prompt and press Enter. Empty line / 'quit' / 'exit' to stop.



Your prompt >  write a java program to find prime number



### Your Prompt: write a java program to find prime number

**Generated Java**
```java
public boolean isPrime(int n) {if (n <= 1)return false;for (int i = 2; i * i <= n; i++)if (n % i == 0)return false;return true;}
```

**Generated C#**
```csharp
public bool IsPrime(int n){if (n <= 1){return false;}for (int i = 2; i * i <= n; i++){if (n % i == 0){return false;}}return true;}
```

> ⏱ Java: **2.77s** | C#: **2.98s** | Total: **5.75s**

---


Your prompt >  /



### Your Prompt: /

**Generated Java**
```java
public static int GetDeltaBaseRevisionCount(RevCommit c){if (c.ParentCount < 2){return -1;}return c.Parent(0).GetDistance(c);}
```

**Generated C#**
```csharp
public static int GetDeltaBaseRevisionCount(RevCommit c){if (c.ParentCount < 2){return -1;}return c.Parent(0).GetDistance(c);}
```

> ⏱ Java: **6.31s** | C#: **2.11s** | Total: **8.42s**

---


Your prompt >  Write a Java program for palindrome



### Your Prompt: Write a Java program for palindrome

**Generated Java**
```java
public boolean isPalindrome(String s) {int i = 0;int j = s.length() - 1;while (i < j) {if (!Character.isLetterOrDigit(s.charAt(i))) {i++;} else if (!Character.isLetterOrDigit(s.charAt(j))) {j--;} else {if (Character.toLowerCase(s.charAt(i)) != Character.toLowerCase(s.charAt(j))) {return false;}i++;j--;}}return true;}
```

**Generated C#**
```csharp
public bool IsPalindrome(string s){for (int i = 0, j = s.Length - 1; i < j; i++, j--){if (!char.IsLetterOrDigit(s[i])){i++;}else if (!char.IsLetterOrDigit(s[j])){j--;}else{if (char.ToLower(s[i]) != char.ToLower(s[j])){return false;}}}return true;}
```

> ⏱ Java: **5.08s** | C#: **4.74s** | Total: **9.82s**

---
